# 04 · Hugging Face dataset

Combines the human, AI-assisted and AI-generated texts into one dataset with three labels, checks it, saves it, and uploads it to the Hugging Face Hub.

## Settings

In [ ]:
import hashlib
import json
import os
from collections import Counter, defaultdict
from pathlib import Path

from datasets import ClassLabel, Dataset, DatasetDict, Features, Value

# This notebook sits in <repo>/04_hf-dataset/, so the repo root is one level up
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data"

HUMAN_DIR = DATA_DIR / "human"
ASSISTED_DIR = DATA_DIR / "ai-assisted"
GENERATED_DIR = DATA_DIR / "ai-generated"
OUTPUT_DIR = DATA_DIR / "hf-dataset"

# Human metadata from step 01 (the first file that exists is used)
HUMAN_META_CANDIDATES = [HUMAN_DIR / "meta.json", HUMAN_DIR / "meta_final.json", HUMAN_DIR / "meta_hf.json"]

# Hugging Face repo to upload to: "<your-hf-username>/<dataset-name>"
HF_REPO_ID = "aliissa824/german-academic-ai-text"

# Label order: from "no AI" to "fully AI"
LABEL_NAMES = ["human", "ai_assisted", "ai_generated"]

# AI texts shorter than this are refusals ("I'm sorry, but I cannot ...")
MIN_AI_WORDS = 20

# Which model handled which human sample IDs (same as in step 02)
# (first_id, last_id, model, company, strategy for the AI-generated text)
GENERATORS = [
    (1, 2499, "gpt-5.6-luna", "OpenAI", "source_based"),
    (2500, 4999, "gemini-3.8-flash", "Google", "source_based"),
    (5000, 7499, "deepseek-flash", "DeepSeek", "topics"),
    (7500, 10227, "gpt-6-luna", "OpenAI", "source_based"),
]

SPLITS = ["train", "validation", "test"]

print("Data folder:", DATA_DIR.relative_to(REPO_ROOT))
print("Labels:     ", LABEL_NAMES)

## Helpers

In [ ]:
FEATURES = Features(
    {
        "id": Value("string"),
        "text": Value("string"),
        "label": ClassLabel(names=LABEL_NAMES),
        "seed_sample_id": Value("int64"),
        "source": Value("string"),
        "source_name": Value("string"),
        "document_id": Value("string"),
        "title": Value("string"),
        "author": Value("string"),
        "publication_year": Value("int32"),
        "document_type": Value("string"),
        "license_name": Value("string"),
        "license_url": Value("string"),
        "source_url": Value("string"),
        "target_length": Value("int32"),
        "word_count": Value("int32"),
        "generator_model": Value("string"),
        "generator_company": Value("string"),
        "strategy": Value("string"),
    }
)


def count_words(text):
    return len(text.split())


def load_human_meta():
    for path in HUMAN_META_CANDIDATES:
        if path.exists():
            samples = json.loads(path.read_text(encoding="utf-8"))["samples"]
            # meta_hf.json also has AI rows: keep only the human ones
            return [s for s in samples if s.get("label") in (0, "human")]
    raise FileNotFoundError(f"No human metadata found in {HUMAN_DIR.relative_to(REPO_ROOT)}")

def human_text_path(sample_id):
    """Human texts may sit directly in data/human/ or in data/human/human/."""
    for path in [HUMAN_DIR / f"{sample_id}.txt", HUMAN_DIR / "human" / f"{sample_id}.txt"]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Human text {sample_id}.txt not found in {HUMAN_DIR}")


def generator_for(seed_id):
    for first, last, model, company, strategy in GENERATORS:
        if first <= seed_id <= last:
            return model, company, strategy
    raise ValueError(f"No generator configured for sample {seed_id}")


def shared_fields(human):
    """Fields every row copies from its human original (incl. attribution)."""
    author = human.get("author")
    if isinstance(author, list):
        author = "; ".join(author)
    date = human.get("publication_date") or ""
    return {
        "source": human.get("source"),
        "source_name": human.get("source_name"),
        "document_id": human.get("document_id"),
        "title": human.get("title"),
        "author": author,
        "publication_year": int(date[:4]) if date[:4].isdigit() else None,
        "document_type": human.get("document_type"),
        "license_name": human.get("license_name"),
        "license_url": human.get("license_url"),
        "source_url": human.get("source_url"),
        "target_length": human.get("target_length"),
    }

## Build

One human row plus its two AI partners, always in the same split. AI texts under 20 words (refusals) and missing files are skipped.

In [ ]:
rows = {split: [] for split in SPLITS}
skipped = defaultdict(list)

for human in load_human_meta():
    seed_id = int(human["sample_id"])
    split = human["split"]
    shared = shared_fields(human)

    # Human row
    text = human_text_path(seed_id).read_text(encoding="utf-8").strip()
    rows[split].append(
        {
            "id": f"human-{seed_id}",
            "text": text,
            "label": 0,
            "seed_sample_id": None,
            **shared,
            "word_count": count_words(text),
            "generator_model": None,
            "generator_company": None,
            "strategy": None,
        }
    )
    human_text = text

    # Its two AI partners, always in the same split
    model, company, generated_strategy = generator_for(seed_id)
    for label, kind, folder, strategy in [
        (1, "assisted", ASSISTED_DIR, "rewrite"),
        (2, "generated", GENERATED_DIR, generated_strategy),
    ]:
        path = folder / f"{seed_id}.txt"
        if not path.exists():
            skipped[f"missing_{kind}"].append(seed_id)
            continue
        text = path.read_text(encoding="utf-8").strip()
        if count_words(text) < MIN_AI_WORDS:
            skipped[f"refused_{kind}"].append(seed_id)
            continue
        if text == human_text:
            skipped[f"unchanged_{kind}"].append(seed_id)
            continue
        rows[split].append(
            {
                "id": f"{kind}-{seed_id}",
                "text": text,
                "label": label,
                "seed_sample_id": seed_id,
                **shared,
                "word_count": count_words(text),
                "generator_model": model,
                "generator_company": company,
                "strategy": strategy,
            }
        )

dataset = DatasetDict(
    {split: Dataset.from_list(rows[split], features=FEATURES) for split in SPLITS}
)

print(dataset)
print("\nSkipped:")
for reason, ids in skipped.items():
    print(f"  {reason}: {len(ids)}  {sorted(ids)[:15]}")

## Check

No document or human/AI group in two splits, no duplicate texts, licence and source on every row.

In [ ]:
errors = []
all_rows = [(split, row) for split in SPLITS for row in dataset[split]]

# Counts per split and label
print(f"{'split':<12}" + "".join(f"{n:>14}" for n in LABEL_NAMES) + f"{'total':>10}")
for split in SPLITS:
    counts = Counter(dataset[split]["label"])
    print(f"{split:<12}" + "".join(f"{counts[i]:>14,}" for i in range(3)) + f"{len(dataset[split]):>10,}")
totals = Counter(row["label"] for _, row in all_rows)
print(f"{'total':<12}" + "".join(f"{totals[i]:>14,}" for i in range(3)) + f"{len(all_rows):>10,}")

# Unique IDs and texts
ids = Counter(row["id"] for _, row in all_rows)
errors += [f"duplicate id {i}" for i, n in ids.items() if n > 1]
hashes = Counter(hashlib.sha256(row["text"].encode()).hexdigest() for _, row in all_rows)
errors += [f"{n} identical texts" for n in hashes.values() if n > 1]

# Row-level checks
doc_splits, group_splits = defaultdict(set), defaultdict(set)
for split, row in all_rows:
    if not row["text"].strip():
        errors.append(f"empty text: {row['id']}")
    if row["word_count"] != count_words(row["text"]):
        errors.append(f"wrong word_count: {row['id']}")
    if not row["license_name"] or not row["source_url"]:
        errors.append(f"missing licence or source URL: {row['id']}")
    if row["label"] == 0 and row["generator_model"] is not None:
        errors.append(f"human row with generator: {row['id']}")
    if row["label"] > 0 and row["word_count"] < MIN_AI_WORDS:
        errors.append(f"AI text too short: {row['id']}")
    doc_splits[row["document_id"]].add(split)
    group_splits[int(row["id"].split("-")[1])].add(split)

# Leakage: a document or a human/AI group in more than one split
leaking_docs = [d for d, s in doc_splits.items() if len(s) > 1]
leaking_groups = [g for g, s in group_splits.items() if len(s) > 1]
print("\nDocuments in more than one split:     ", len(leaking_docs))
print("Human/AI groups in more than one split:", len(leaking_groups))
print("Identical texts:                       ", sum(n > 1 for n in hashes.values()))
if leaking_docs or leaking_groups:
    errors.append("split leakage")

if errors:
    raise ValueError(f"{len(errors)} problems, e.g. {errors[:10]}")
print("\nAll checks passed.")

## Save

Saves to `data/hf-dataset/` and writes the dataset card with the real counts.

In [ ]:
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
dataset.save_to_disk(str(OUTPUT_DIR))

# Fill the dataset card (dataset_card.md, next to this notebook) with the real counts
lines = []
for split in SPLITS:
    counts = Counter(dataset[split]["label"])
    lines.append(f"| {split} | " + " | ".join(f"{counts[i]:,}" for i in range(3)) + f" | {len(dataset[split]):,} |")
totals = Counter(label for split in SPLITS for label in dataset[split]["label"])
total = sum(len(dataset[split]) for split in SPLITS)
lines.append("| **total** | " + " | ".join(f"**{totals[i]:,}**" for i in range(3)) + f" | **{total:,}** |")

card = Path("dataset_card.md").read_text(encoding="utf-8")
card = card.replace("{{COUNTS_TABLE}}", "\n".join(lines)).replace("{{HF_REPO_ID}}", HF_REPO_ID)
(OUTPUT_DIR / "README.md").write_text(card, encoding="utf-8")

print("Saved to", OUTPUT_DIR)

## Examples for the README

In [ ]:
wanted = {f"{k}-{i}" for i in (697, 2884, 7164, 9857) for k in ("human", "assisted", "generated")}

print("| id | label | source | word_count | generator_model | strategy | license_name |")
print("|---|---|---|---|---|---|---|")
for split in SPLITS:
    for row in dataset[split]:
        if row["id"] in wanted:
            print(
                f"| {row['id']} | {LABEL_NAMES[row['label']]} | {row['source']} "
                f"| {row['word_count']} | {row['generator_model'] or '–'} "
                f"| {row['strategy'] or '–'} | {row['license_name']} |"
            )

## Upload

Set `HF_REPO_ID` above and `HF_TOKEN` in `.env`, then set `UPLOAD = True`.

In [ ]:
# Needs HF_TOKEN (write access) in the .env file in the repo root.
# Starts private: check the page on the Hub, then make it public there.
UPLOAD = False

if UPLOAD:
    from dotenv import load_dotenv
    from huggingface_hub import DatasetCard

    load_dotenv(REPO_ROOT / ".env", override=True)
    token = os.environ["HF_TOKEN"]
    dataset.push_to_hub(HF_REPO_ID, private=True, token=token)
    DatasetCard((OUTPUT_DIR / "README.md").read_text(encoding="utf-8")).push_to_hub(HF_REPO_ID, token=token)
    print(f"Uploaded: https://huggingface.co/datasets/{HF_REPO_ID}")